# Cortical TRF analysis 2: neural speech restoration at the cocktail party

This notebook runs the same kinds of analyses as Brodbeck et al. 2020
(PLOS Biology), "Neural speech restoration at the cocktail party", on this
project's own EEG dataset (13 subjects) instead of their 26-subject MEG
dataset. Cells that match one of their reported comparisons carry a short
comment naming the result they found, as a point of reference - not a
target this dataset is expected to hit. Different dataset, different
modality, a much smaller group: matching the *methodology* is the goal,
not reproducing their exact numbers.

**Renaming from `cortical_analysis.ipynb`:** foreground -> **attend**,
background -> **ignore**, mixture stays **mix**. This only changes labels
and variable names in this notebook - `experiment.py`'s `label_events()`
still populates `ds['fg']`/`ds['bg']`/`ds['mix']` under those original
names, shared with the other notebook.

**Frequency resolution:** 8-band predictors (`gammatone-8`,
`gammatone-on-8`) throughout, not the 1-band versions
`cortical_analysis.ipynb` mostly uses, so TRFs can be examined per
frequency band (an STRF - spectro-temporal response function).

## Structure

Organized by listening condition, each as its own section: clean speech,
then diotic, dichotic-left, dichotic-right, and binaural - each of the
four two-talker conditions further split into its own base section plus
overt-onset, masked-onset, and intensity-level-control subsections. Every
subsection is self-contained: the same handful of shared helper functions
(defined once below) get called with that subsection's own predictors and
`epoch`. Cells run top to bottom; `data`, `diff`, `result`, `p` are reused
generically across subsections the same way `cortical_analysis.ipynb`
reuses `data`/`p`/`dss` - what a cell overwrites is always exactly what
the cell right before it just produced.

In [ ]:
from eelbrain import *
from experiment import e, topomap_with_colorbar, FIGURES_DIR

# Where to save plots - inside the BIDS dataset's derivatives, same as
# cortical_analysis.ipynb, not somewhere outside the dataset.
DST = FIGURES_DIR

## Preprocessing (run once per subject)

Identical to `cortical_analysis.ipynb` - both notebooks use the same
preprocessed EEG (`raw='ica'`), so this only does real work the first time
either notebook is run for a given subject.

In [ ]:
MANUAL_BAD_CHANNELS = False
AUTO_BAD_CHANNELS_R = 0.3
REDO_BAD_CHANNELS = True

e.mark_bad_channels(
    'all',
    manual_bad_channels=MANUAL_BAD_CHANNELS, auto_bad_channels_r=AUTO_BAD_CHANNELS_R,
    redo_bad_channels=REDO_BAD_CHANNELS,
)

In [ ]:
MANUAL_ICA = False
AUTO_ICA_CONFIDENCE = 0.75

e.select_ica_artifacts(
    'all',
    manual_ica=MANUAL_ICA, auto_ica_confidence=AUTO_ICA_CONFIDENCE,
)

## Shared setup

Everything in this section is reused by every condition section below.

### Stimulus-role and predictor-name helpers

Every predictor is built from the same handful of pieces: envelope or
onset, attended/ignored/mixture, and (for the masking/intensity
subsections) overt/masked/loudness-tier splits. A "stimulus role" prefix
picks which event-dataset column supplies the stimulus name for a term -
this is the same `bg~`/`mix~` mechanism `cortical_analysis.ipynb` already
uses, just wrapped in small functions so model strings don't have to be
retyped by hand:

- `''` (empty) = attended speaker (`fg` column, `experiment.py`'s default
  `stim_var` - no prefix needed)
- `'bg~'` = ignored speaker (`bg` column)
- `'mix~'` = the acoustic mixture (`mix` column)

The intensity-tier predictors are named `gammatone-on-low-8`/`-mid-8`/
`-high-8` rather than "mixture-onset-low": eelbrain only recognizes
`gammatone` as a registered predictor key (see `predictors` in
`experiment.py`), so every predictor code has to start with `gammatone-`
to resolve at all - see the comment above that block in
`predictors/gammatone_predictors.py`.

In [ ]:
ATTEND, IGNORE, MIX = '', 'bg~', 'mix~'


def envelope(role=ATTEND):
    return f"{role}gammatone-8"


def onset(role=ATTEND):
    return f"{role}gammatone-on-8"


def overt_onset(role=ATTEND):
    return f"{role}gammatone-on-overt-8"


def masked_onset(role=ATTEND):
    return f"{role}gammatone-on-masked-8"


def level_onset(tier, role=MIX):
    assert tier in ('low', 'mid', 'high')
    return f"{role}gammatone-on-{tier}-8"


# The three model-comparisons every condition section below is built
# from. Only `epoch=` changes between diotic/dichotic-left/dichotic-right/
# binaural - the model strings themselves don't depend on condition.
FULL_CLEAN = f"{envelope(ATTEND)} + {onset(ATTEND)}"
FULL_BASE = f"{envelope(ATTEND)} + {onset(ATTEND)} + {envelope(IGNORE)} + {onset(IGNORE)} + {envelope(MIX)} + {onset(MIX)}"
FULL_MASKED = (
    f"{envelope(ATTEND)} + {overt_onset(ATTEND)} + {masked_onset(ATTEND)} + "
    f"{envelope(IGNORE)} + {overt_onset(IGNORE)} + {masked_onset(IGNORE)} + "
    f"{envelope(MIX)} + {onset(MIX)}"
)
LEVEL_AWARE = (
    f"{envelope(ATTEND)} + {onset(ATTEND)} + "
    f"{envelope(IGNORE)} + "
    f"{envelope(MIX)} + {level_onset('low')} + {level_onset('mid')} + {level_onset('high')}"
)

### Fit metric: correlation (r), not proportion of variance explained (ev)

`METRIC` is the one place this choice is made - every model comparison
below reads it from here rather than hard-coding `'ev'` or `'r'`. Change
it here to switch every analysis in this notebook at once.

In [ ]:
METRIC = 'r'  # 'r' (Pearson correlation) or 'ev' (proportion of variance explained)
PMIN = 0.05

# ev is a tiny fraction of variance (needs a fixed, narrow color scale to
# stay readable); r has a much larger natural range and reads fine with
# an automatic scale - see cortical_analysis.ipynb's own TOPO_ARGS/
# TOPO_ARGS_R split for the same reasoning.
TOPO_ARGS = dict(clip='circle') if METRIC == 'r' else dict(vmax=0.005, clip='circle')

### Shared TRF-fitting settings and the region of interest (ROI)

`PARAMETERS` is identical to `cortical_analysis.ipynb`. `ROI` is the same
fronto-central sensor set too - not a new one.

In [ ]:
PARAMETERS = {
    'raw': 'ica',
    'group': 'all',
    'samplingrate': 128,
    'data': 'eeg',
    'tstart': -0.100,
    'tstop': 0.600,
}

data = e.load_trfs(1, 'gammatone-8', epoch='clean', **PARAMETERS)
eeg = data['ev']

ROI = [
    'AF3', 'F1', 'F3', 'F5',
    'FC5', 'FC3', 'FC1',
    'C1', 'C3', 'C5',
    'AF4', 'AFz',
    'Fz', 'F2', 'F4', 'F6',
    'FC6', 'FC4', 'FC2', 'FCz',
    'Cz', 'C2', 'C4', 'C6',
]

p = plot.SensorMap(eeg, mark=ROI)

### Display names and colors for attend/ignore/mix

Reused everywhere a plot needs to tell the three streams apart at a
glance - same red/blue/gray convention `cortical_analysis.ipynb` uses for
foreground/background/mixture.

In [ ]:
LABELS_ROLE = {'attend': 'Attend', 'ignore': 'Ignore', 'mix': 'Mixture'}
COLORS_ROLE = {'attend': 'red', 'ignore': 'blue', 'mix': '.3'}

### Peak-timing search windows

Named here, once, so every "peak amplitude & timing" cell below points at
one of these instead of a bare number buried in the cell - change a
window in one place and every analysis that uses it picks it up. Onset
and envelope get their own windows; overt/masked onsets get wider ones
since masked onsets are expected to peak later than overt ones (Brodbeck:
roughly 15-50ms slower).

In [ ]:
ONSET_POSITIVE_PEAK_WINDOW = (0.020, 0.130)     # Brodbeck onset positive peak: ~65ms
ONSET_NEGATIVE_PEAK_WINDOW = (0.080, 0.200)     # Brodbeck onset negative peak: ~126ms
ENVELOPE_POSITIVE_PEAK_WINDOW = (0.020, 0.200)  # envelope responses are weaker/less sharply peaked
ENVELOPE_NEGATIVE_PEAK_WINDOW = (0.100, 0.300)
MASKED_POSITIVE_PEAK_WINDOW = (0.020, 0.160)    # wider: also used for the overt-vs-masked comparison
MASKED_NEGATIVE_PEAK_WINDOW = (0.100, 0.250)

### Model-comparison helpers

Two comparison syntaxes are used below (see `e.load_model_test`'s own
docstring): **`full_model @ predictor`** (omission - fit `full_model`,
then a null model with just `predictor` removed) for "unique
contribution" analyses, and **`model_a > model_b`** (a *direct* comparison
between two independently-specified, non-nested models) for the
overt/masked-split-vs-plain-onsets and masked-aware-vs-level-aware
comparisons - eelbrain supports this directly in its model-string
grammar, no separate API needed.

Both return the same shape of result: `data` (one row per subject per
model - `'test'`/`'baseline'`), `diff` (one row per subject: the
per-subject improvement score, `test` minus `baseline`, already averaged
over the ROI), and `result` (the mass-univariate significance-test
object). Each condition section below calls these once per analysis, then
uses `data`/`diff`/`result` across the next couple of cells - the topomap
trio, the stat result, and the per-subject bar plot are deliberately
separate cells so each one's output is easy to find.

In [ ]:
def compute_unique_contribution(full_model, predictor, epoch, roi=ROI, metric=METRIC):
    "Omission comparison: does `predictor` uniquely improve on `full_model` with it removed?"
    comparison = f"{full_model} @ {predictor}"
    data, result = e.load_model_test(comparison, **PARAMETERS, pmin=PMIN, metric=metric, epoch=epoch, return_data=True)
    diff = table.difference(metric, 'model', 'test', 'baseline', 'subject', data=data)
    diff[f'roi_{metric}'] = diff[metric].mean(sensor=roi)
    return data, diff, result


def compute_direct_comparison(model_a, model_b, epoch, roi=ROI, metric=METRIC):
    "Direct, non-nested comparison: does model_a fit better than model_b?"
    comparison = f"{model_a} > {model_b}"
    data, result = e.load_model_test(comparison, **PARAMETERS, pmin=PMIN, metric=metric, epoch=epoch, return_data=True)
    diff = table.difference(metric, 'model', 'test', 'baseline', 'subject', data=data)
    diff[f'roi_{metric}'] = diff[metric].mean(sensor=roi)
    return data, diff, result


def show_topomap_trio(data, result, label, epoch, predictor, metric=METRIC):
    "Group-averaged topomaps: the test model's own fit, the baseline model's own fit, and their significance-masked difference"
    pct = metric == 'ev'
    topo_kwargs = dict(pct=pct, condition=epoch, predictor=predictor, colorbar_side='left', colorbar_label=False, **TOPO_ARGS)
    topomap_with_colorbar(metric, data.sub("model == 'test'"), label=f'{label}: test', **topo_kwargs)
    topomap_with_colorbar(metric, data.sub("model == 'baseline'"), label=f'{label}: baseline', **topo_kwargs)
    # The third panel plots the significance-masked map (not the plain
    # mean difference): topomap_with_colorbar accepts a bare NDVar for
    # exactly this - the masked-out (non-significant) sensors show up as
    # a visible boundary around the surviving, significant region.
    masked_diff = result.masked_parameter_map(pmin=PMIN)
    topomap_with_colorbar(None, masked_diff, label=f'{label}: test - baseline', **topo_kwargs)


def show_subject_bar(diff, label, epoch, metric=METRIC):
    "Per-subject bar plot of the ROI-averaged improvement score - every subject's own value, not just the group mean"
    return plot.Barplot(f'roi_{metric}', match='subject', data=diff, h=2.5, w=2, corr=False, title=f'{label} ({epoch})')

### STRF and peak-timing helpers

`roi_strf` gives the group-average (frequency, time) response for one
predictor, for the frequency-resolved STRF plots. `roi_trf`/`add_peak`
give the per-subject, ROI-averaged, frequency-summed response used for
every peak-amplitude/timing extraction below - `add_peak` is deliberately
generic (a name and a window in, two new columns out) so it's the same
one function for onsets, envelope, overt, and masked alike.

In [ ]:
def roi_strf(predictor_model, predictor_column, epoch, roi=ROI):
    "Group-average (frequency, time) STRF for one predictor - the ROI/subject average of its fitted TRF"
    data = e.load_trfs('all', predictor_model, epoch=epoch, **PARAMETERS)
    return data[predictor_column].mean(sensor=roi).mean('case')


def roi_trf(model, column, epoch, roi=ROI):
    "Per-subject, ROI-averaged, frequency-summed, smoothed TRF for one predictor"
    data = e.load_trfs(-1, model, epoch=epoch, **PARAMETERS)
    trf = data[column].sum('frequency').mean(sensor=roi)
    ds = data['subject',]
    ds['trf_sm'] = resample(trf, 512)
    return ds


def add_peak(ds, name, window, kind='max'):
    "Add `{name}_time`/`{name}_amp`: the location/value of ds['trf_sm']'s peak within `window`"
    segment = ds['trf_sm'].sub(time=window)
    if kind == 'max':
        ds[f'{name}_time'] = segment.argmax('time')
        ds[f'{name}_amp'] = segment.max('time')
    else:
        ds[f'{name}_time'] = segment.argmin('time')
        ds[f'{name}_amp'] = segment.min('time')
    return ds

## Clean speech (single speaker)

In [ ]:
# Onset unique contribution (clean)
# Brodbeck et al. 2020 ran this comparison too: single-speaker onset unique contribution, t = 12.00, p <= .001.
data, diff, result = compute_unique_contribution(FULL_CLEAN, onset(ATTEND), 'clean')
show_topomap_trio(data, result, 'Onset', 'clean', onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset (clean): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset', 'clean')

In [ ]:
# Envelope unique contribution (clean)
# Brodbeck et al. 2020 ran this comparison too: single-speaker envelope unique contribution, t = 9.39, p <= .001.
data, diff, result = compute_unique_contribution(FULL_CLEAN, envelope(ATTEND), 'clean')
show_topomap_trio(data, result, 'Envelope', 'clean', envelope(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope (clean): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope', 'clean')

In [ ]:
# Onset STRF (clean)
strf = roi_strf(FULL_CLEAN, 'gammatone_on_8', 'clean')
p = plot.Array(strf * 1e3, title='Onset STRF (clean)', xlim=(-0.050, 0.550))

In [ ]:
# Envelope STRF (clean)
strf = roi_strf(FULL_CLEAN, 'gammatone_8', 'clean')
p = plot.Array(strf * 1e3, title='Envelope STRF (clean)', xlim=(-0.050, 0.550))

In [ ]:
# Onset peak amplitude & timing (clean)
# Brodbeck et al. 2020 ran this comparison too: onset STRF positive peak ~65ms, negative peak ~126ms.
data = roi_trf(FULL_CLEAN, 'gammatone_on_8', 'clean')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset (clean) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset (clean) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# The TRF waveform the peaks above were measured from - same
# smoothed, ROI-averaged, frequency-summed response as 'data'.
p = plot.UTSStat('trf_sm*1e3', data=data, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Onset TRF (clean)')

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset positive peak time (clean)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset positive peak amplitude (clean)')

In [ ]:
# Envelope peak amplitude & timing (clean)
data = roi_trf(FULL_CLEAN, 'gammatone_8', 'clean')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope (clean) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope (clean) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# The TRF waveform the peaks above were measured from - same
# smoothed, ROI-averaged, frequency-summed response as 'data'.
p = plot.UTSStat('trf_sm*1e3', data=data, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Envelope TRF (clean)')

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope positive peak time (clean)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope positive peak amplitude (clean)')

## Diotic

In [ ]:
# Onset attend unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: attended onsets unique contribution, t = 6.32, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(ATTEND), 'diotic')
show_topomap_trio(data, result, 'Onset attend', 'diotic', onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset attend (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset attend', 'diotic')

In [ ]:
# Onset ignore unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: ignored onsets unique contribution, t = 6.70, p < .001 - the ignored speaker is tracked separately.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(IGNORE), 'diotic')
show_topomap_trio(data, result, 'Onset ignore', 'diotic', onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset ignore (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset ignore', 'diotic')

In [ ]:
# Onset mix unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: mixture onsets unique contribution, t = 8.61, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(MIX), 'diotic')
show_topomap_trio(data, result, 'Onset mix', 'diotic', onset(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset mix (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset mix', 'diotic')

In [ ]:
# Envelope attend unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: attended envelope unique contribution, t = 7.37, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(ATTEND), 'diotic')
show_topomap_trio(data, result, 'Envelope attend', 'diotic', envelope(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope attend (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope attend', 'diotic')

In [ ]:
# Envelope ignore unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: ignored envelope unique contribution, t = 6.28, p < .001 - the ignored speaker's envelope is tracked separately too.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(IGNORE), 'diotic')
show_topomap_trio(data, result, 'Envelope ignore', 'diotic', envelope(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope ignore (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope ignore', 'diotic')

In [ ]:
# Envelope mix unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: mixture envelope unique contribution, t = 5.70, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(MIX), 'diotic')
show_topomap_trio(data, result, 'Envelope mix', 'diotic', envelope(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope mix (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope mix', 'diotic')

In [ ]:
# Mix source STRF: envelope + onset (diotic)
envelope_strf = roi_strf(FULL_BASE, 'mix_gammatone_8', 'diotic')
onset_strf = roi_strf(FULL_BASE, 'mix_gammatone_on_8', 'diotic')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Mix source STRF (diotic)', xlim=(-0.050, 0.550))

In [ ]:
# Attend source STRF: envelope + onset (diotic)
envelope_strf = roi_strf(FULL_BASE, 'gammatone_8', 'diotic')
onset_strf = roi_strf(FULL_BASE, 'gammatone_on_8', 'diotic')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Attend source STRF (diotic)', xlim=(-0.050, 0.550))

In [ ]:
# Ignore source STRF: envelope + onset (diotic)
envelope_strf = roi_strf(FULL_BASE, 'bg_gammatone_8', 'diotic')
onset_strf = roi_strf(FULL_BASE, 'bg_gammatone_on_8', 'diotic')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Ignore source STRF (diotic)', xlim=(-0.050, 0.550))

In [ ]:
# Onset attend peak amplitude & timing (diotic)
data = roi_trf(FULL_BASE, 'gammatone_on_8', 'diotic')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset attend (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset attend (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak amplitude (diotic)')

In [ ]:
# Onset ignore peak amplitude & timing (diotic)
data = roi_trf(FULL_BASE, 'bg_gammatone_on_8', 'diotic')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset ignore (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset ignore (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak amplitude (diotic)')

In [ ]:
# Onset mix peak amplitude & timing (diotic)
data = roi_trf(FULL_BASE, 'mix_gammatone_on_8', 'diotic')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset mix (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset mix (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak amplitude (diotic)')

In [ ]:
# Envelope attend peak amplitude & timing (diotic)
data = roi_trf(FULL_BASE, 'gammatone_8', 'diotic')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope attend (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope attend (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak amplitude (diotic)')

In [ ]:
# Envelope ignore peak amplitude & timing (diotic)
data = roi_trf(FULL_BASE, 'bg_gammatone_8', 'diotic')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope ignore (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope ignore (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak amplitude (diotic)')

In [ ]:
# Envelope mix peak amplitude & timing (diotic)
data = roi_trf(FULL_BASE, 'mix_gammatone_8', 'diotic')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope mix (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope mix (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak amplitude (diotic)')

In [ ]:
# Onset peak comparison across mix, attend, and ignore (diotic)
# Brodbeck et al. 2020 ran this comparison too: positive peak latency mixture 72ms, attended 81ms (t=4.47,p<.001), ignored 89ms (t=6.92,p<.001); negative peak latency mixture 138ms, attended 150ms (t=3.20,p=.004); positive peak amplitude bigger for mixture than attended (t=8.41) or ignored (t=7.66).
dss = []
for role_name, column in (('mix', 'mix_gammatone_on_8'), ('attend', 'gammatone_on_8'), ('ignore', 'bg_gammatone_on_8')):
    ds = roi_trf(FULL_BASE, column, 'diotic')
    ds[:, 'role'] = role_name
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
# The onset TRF waveform mix/attend/ignore were each just measured
# from, overlaid so their shape and relative timing are visible
# directly, not just as extracted peak numbers below.
p = plot.UTSStat('trf_sm*1e3', 'role', data=data, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550),
                  title='Onset TRF: mix vs. attend vs. ignore (diotic)', labels=LABELS_ROLE, colors=COLORS_ROLE)

In [ ]:
display(test.pairwise('positive_time', 'role', match='subject', data=data, corr=False, title='positive peak latency: mix vs. attend vs. ignore'))
display(test.pairwise('negative_time', 'role', match='subject', data=data, corr=False, title='negative peak latency: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak latency (diotic)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='negative peak latency (diotic)')
p.set_xtick_rotation(30)

In [ ]:
display(test.pairwise('positive_amp', 'role', match='subject', data=data, corr=False, title='positive peak amplitude: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak amplitude (diotic)')
p.set_xtick_rotation(30)

In [ ]:
# Envelope attention effect: attend - ignore (diotic)
# Brodbeck et al. 2020 ran this comparison too: envelope attention-effect difference wave: negative bump ~100ms, positive bump ~200ms, starting almost immediately.
data = e.load_trfs(-1, FULL_BASE, epoch='diotic', **PARAMETERS)
attend_trf = data['gammatone_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Envelope attention effect: attend - ignore (diotic)')

In [ ]:
# Onset attention effect: attend - ignore (diotic)
# Brodbeck et al. 2020 ran this comparison too: onset attention effect appears later than the envelope's, once the masked-onset response develops (see the masked-onset section below).
data = e.load_trfs(-1, FULL_BASE, epoch='diotic', **PARAMETERS)
attend_trf = data['gammatone_on_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Onset attention effect: attend - ignore (diotic)')

## Diotic (overt onsets)

In [ ]:
# Overt/masked split vs. plain onsets (diotic)
# Brodbeck et al. 2020 ran this comparison too: splitting onsets into overt/masked significantly improved the model overall, t = 6.81, p < .001.
data, diff, result = compute_direct_comparison(FULL_MASKED, FULL_BASE, 'diotic')
show_topomap_trio(data, result, 'Overt/masked split vs. plain onsets', 'diotic', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt/masked split vs. plain onsets (diotic): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt/masked split vs. plain onsets', 'diotic')

In [ ]:
# Overt onset attend unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: attended overt onsets unique contribution, t = 3.34, p = .027.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(ATTEND), 'diotic')
show_topomap_trio(data, result, 'Overt onset attend', 'diotic', overt_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset attend (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset attend', 'diotic')

In [ ]:
# Overt onset ignore unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: ignored overt onsets unique contribution, t = 3.82, p = .016.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(IGNORE), 'diotic')
show_topomap_trio(data, result, 'Overt onset ignore', 'diotic', overt_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset ignore (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset ignore', 'diotic')

In [ ]:
# Overt onset attend STRF (diotic)
strf = roi_strf(FULL_MASKED, 'gammatone_on_overt_8', 'diotic')
p = plot.Array(strf * 1e3, title='Overt onset attend STRF (diotic)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset ignore STRF (diotic)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'diotic')
p = plot.Array(strf * 1e3, title='Overt onset ignore STRF (diotic)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset attend peak amplitude & timing (diotic)
data = roi_trf(FULL_MASKED, 'gammatone_on_overt_8', 'diotic')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset attend (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset attend (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak amplitude (diotic)')

In [ ]:
# Overt onset ignore peak amplitude & timing (diotic)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'diotic')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset ignore (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset ignore (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak amplitude (diotic)')

In [ ]:
# Overt onset attention effect: attend - ignore (diotic)
data = e.load_trfs(-1, FULL_MASKED, epoch='diotic', **PARAMETERS)
attend_trf = data['gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Overt onset attention effect: attend - ignore (diotic)')

## Diotic (masked onset)

In [ ]:
# Masked onset attend unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: attended masked onsets unique contribution, t = 8.42, p < .001 - the strongest of the four masking predictors.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(ATTEND), 'diotic')
show_topomap_trio(data, result, 'Masked onset attend', 'diotic', masked_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset attend (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset attend', 'diotic')

In [ ]:
# Masked onset ignore unique contribution (diotic)
# Brodbeck et al. 2020 ran this comparison too: ignored masked onsets unique contribution, t = 5.23, p < .001 - the brain recovers masked information even for the ignored speaker.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(IGNORE), 'diotic')
show_topomap_trio(data, result, 'Masked onset ignore', 'diotic', masked_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset ignore (diotic): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset ignore', 'diotic')

In [ ]:
# Masked onset attend STRF (diotic)
strf = roi_strf(FULL_MASKED, 'gammatone_on_masked_8', 'diotic')
p = plot.Array(strf * 1e3, title='Masked onset attend STRF (diotic)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset ignore STRF (diotic)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'diotic')
p = plot.Array(strf * 1e3, title='Masked onset ignore STRF (diotic)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset attend peak amplitude & timing (diotic)
data = roi_trf(FULL_MASKED, 'gammatone_on_masked_8', 'diotic')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset attend (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset attend (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak amplitude (diotic)')

In [ ]:
# Masked onset ignore peak amplitude & timing (diotic)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'diotic')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset ignore (diotic) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset ignore (diotic) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak time (diotic)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak amplitude (diotic)')

In [ ]:
# Masked onset attention effect: attend - ignore (diotic)
data = e.load_trfs(-1, FULL_MASKED, epoch='diotic', **PARAMETERS)
attend_trf = data['gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Masked onset attention effect: attend - ignore (diotic)')

In [ ]:
# Overt vs. masked peak comparison, attend (diotic)
# Brodbeck et al. 2020 ran this comparison too: attended overt 72ms vs. attended masked 91ms (t=2.85,p=.009); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'gammatone_on_overt_8'), ('masked', 'gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'diotic')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='attend positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='attend negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='attend positive peak amplitude: overt vs. masked'))

In [ ]:
# Overt vs. masked peak comparison, ignore (diotic)
# Brodbeck et al. 2020 ran this comparison too: ignored overt 83ms vs. ignored masked 97ms (t=6.11,p<.001); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'bg_gammatone_on_overt_8'), ('masked', 'bg_gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'diotic')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='ignore positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='ignore negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='ignore positive peak amplitude: overt vs. masked'))

In [ ]:
# Stream x masking interaction on peak amplitude (diotic)
# Brodbeck et al. 2020 ran this comparison too: stream x masking interaction, F(1,25) = 24.45, p < .001 - overt onsets show no early attention effect, masked onsets do.
dss = []
for stream_name, is_attend in (('attend', True), ('ignore', False)):
    for masking_name in ('overt', 'masked'):
        prefix = '' if is_attend else 'bg_'
        column = f'{prefix}gammatone_on_{masking_name}_8'
        ds = roi_trf(FULL_MASKED, column, 'diotic')
        ds[:, 'stream'] = stream_name
        ds[:, 'masking'] = masking_name
        dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
display(test.ANOVA('positive_amp', 'stream * masking * subject', data=data, title='stream x masking interaction (diotic)'))

## Diotic (intensity-level control)

In [ ]:
# Masked-aware vs. level-aware (diotic)
# Brodbeck et al. 2020 ran this comparison too: the ignored-speaker-aware model still beat the loudness-only model, t = 9.21, p < .001 - the masking effect isn't just about volume.
data, diff, result = compute_direct_comparison(FULL_MASKED, LEVEL_AWARE, 'diotic')
show_topomap_trio(data, result, 'Masked-aware vs. level-aware', 'diotic', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked-aware vs. level-aware (diotic): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked-aware vs. level-aware', 'diotic')

## Dichotic Left

In [ ]:
# Onset attend unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: attended onsets unique contribution, t = 6.32, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(ATTEND), 'dichotic-left')
show_topomap_trio(data, result, 'Onset attend', 'dichotic-left', onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset attend (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset attend', 'dichotic-left')

In [ ]:
# Onset ignore unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: ignored onsets unique contribution, t = 6.70, p < .001 - the ignored speaker is tracked separately.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(IGNORE), 'dichotic-left')
show_topomap_trio(data, result, 'Onset ignore', 'dichotic-left', onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset ignore (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset ignore', 'dichotic-left')

In [ ]:
# Onset mix unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: mixture onsets unique contribution, t = 8.61, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(MIX), 'dichotic-left')
show_topomap_trio(data, result, 'Onset mix', 'dichotic-left', onset(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset mix (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset mix', 'dichotic-left')

In [ ]:
# Envelope attend unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: attended envelope unique contribution, t = 7.37, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(ATTEND), 'dichotic-left')
show_topomap_trio(data, result, 'Envelope attend', 'dichotic-left', envelope(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope attend (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope attend', 'dichotic-left')

In [ ]:
# Envelope ignore unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: ignored envelope unique contribution, t = 6.28, p < .001 - the ignored speaker's envelope is tracked separately too.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(IGNORE), 'dichotic-left')
show_topomap_trio(data, result, 'Envelope ignore', 'dichotic-left', envelope(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope ignore (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope ignore', 'dichotic-left')

In [ ]:
# Envelope mix unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: mixture envelope unique contribution, t = 5.70, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(MIX), 'dichotic-left')
show_topomap_trio(data, result, 'Envelope mix', 'dichotic-left', envelope(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope mix (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope mix', 'dichotic-left')

In [ ]:
# Mix source STRF: envelope + onset (dichotic-left)
envelope_strf = roi_strf(FULL_BASE, 'mix_gammatone_8', 'dichotic-left')
onset_strf = roi_strf(FULL_BASE, 'mix_gammatone_on_8', 'dichotic-left')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Mix source STRF (dichotic-left)', xlim=(-0.050, 0.550))

In [ ]:
# Attend source STRF: envelope + onset (dichotic-left)
envelope_strf = roi_strf(FULL_BASE, 'gammatone_8', 'dichotic-left')
onset_strf = roi_strf(FULL_BASE, 'gammatone_on_8', 'dichotic-left')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Attend source STRF (dichotic-left)', xlim=(-0.050, 0.550))

In [ ]:
# Ignore source STRF: envelope + onset (dichotic-left)
envelope_strf = roi_strf(FULL_BASE, 'bg_gammatone_8', 'dichotic-left')
onset_strf = roi_strf(FULL_BASE, 'bg_gammatone_on_8', 'dichotic-left')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Ignore source STRF (dichotic-left)', xlim=(-0.050, 0.550))

In [ ]:
# Onset attend peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_BASE, 'gammatone_on_8', 'dichotic-left')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset attend (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset attend (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak amplitude (dichotic-left)')

In [ ]:
# Onset ignore peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_BASE, 'bg_gammatone_on_8', 'dichotic-left')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset ignore (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset ignore (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak amplitude (dichotic-left)')

In [ ]:
# Onset mix peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_BASE, 'mix_gammatone_on_8', 'dichotic-left')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset mix (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset mix (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak amplitude (dichotic-left)')

In [ ]:
# Envelope attend peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_BASE, 'gammatone_8', 'dichotic-left')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope attend (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope attend (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak amplitude (dichotic-left)')

In [ ]:
# Envelope ignore peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_BASE, 'bg_gammatone_8', 'dichotic-left')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope ignore (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope ignore (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak amplitude (dichotic-left)')

In [ ]:
# Envelope mix peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_BASE, 'mix_gammatone_8', 'dichotic-left')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope mix (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope mix (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak amplitude (dichotic-left)')

In [ ]:
# Onset peak comparison across mix, attend, and ignore (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: positive peak latency mixture 72ms, attended 81ms (t=4.47,p<.001), ignored 89ms (t=6.92,p<.001); negative peak latency mixture 138ms, attended 150ms (t=3.20,p=.004); positive peak amplitude bigger for mixture than attended (t=8.41) or ignored (t=7.66).
dss = []
for role_name, column in (('mix', 'mix_gammatone_on_8'), ('attend', 'gammatone_on_8'), ('ignore', 'bg_gammatone_on_8')):
    ds = roi_trf(FULL_BASE, column, 'dichotic-left')
    ds[:, 'role'] = role_name
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
# The onset TRF waveform mix/attend/ignore were each just measured
# from, overlaid so their shape and relative timing are visible
# directly, not just as extracted peak numbers below.
p = plot.UTSStat('trf_sm*1e3', 'role', data=data, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550),
                  title='Onset TRF: mix vs. attend vs. ignore (dichotic-left)', labels=LABELS_ROLE, colors=COLORS_ROLE)

In [ ]:
display(test.pairwise('positive_time', 'role', match='subject', data=data, corr=False, title='positive peak latency: mix vs. attend vs. ignore'))
display(test.pairwise('negative_time', 'role', match='subject', data=data, corr=False, title='negative peak latency: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak latency (dichotic-left)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='negative peak latency (dichotic-left)')
p.set_xtick_rotation(30)

In [ ]:
display(test.pairwise('positive_amp', 'role', match='subject', data=data, corr=False, title='positive peak amplitude: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak amplitude (dichotic-left)')
p.set_xtick_rotation(30)

In [ ]:
# Envelope attention effect: attend - ignore (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: envelope attention-effect difference wave: negative bump ~100ms, positive bump ~200ms, starting almost immediately.
data = e.load_trfs(-1, FULL_BASE, epoch='dichotic-left', **PARAMETERS)
attend_trf = data['gammatone_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Envelope attention effect: attend - ignore (dichotic-left)')

In [ ]:
# Onset attention effect: attend - ignore (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: onset attention effect appears later than the envelope's, once the masked-onset response develops (see the masked-onset section below).
data = e.load_trfs(-1, FULL_BASE, epoch='dichotic-left', **PARAMETERS)
attend_trf = data['gammatone_on_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Onset attention effect: attend - ignore (dichotic-left)')

## Dichotic Left (overt onsets)

In [ ]:
# Overt/masked split vs. plain onsets (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: splitting onsets into overt/masked significantly improved the model overall, t = 6.81, p < .001.
data, diff, result = compute_direct_comparison(FULL_MASKED, FULL_BASE, 'dichotic-left')
show_topomap_trio(data, result, 'Overt/masked split vs. plain onsets', 'dichotic-left', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt/masked split vs. plain onsets (dichotic-left): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt/masked split vs. plain onsets', 'dichotic-left')

In [ ]:
# Overt onset attend unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: attended overt onsets unique contribution, t = 3.34, p = .027.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(ATTEND), 'dichotic-left')
show_topomap_trio(data, result, 'Overt onset attend', 'dichotic-left', overt_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset attend (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset attend', 'dichotic-left')

In [ ]:
# Overt onset ignore unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: ignored overt onsets unique contribution, t = 3.82, p = .016.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(IGNORE), 'dichotic-left')
show_topomap_trio(data, result, 'Overt onset ignore', 'dichotic-left', overt_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset ignore (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset ignore', 'dichotic-left')

In [ ]:
# Overt onset attend STRF (dichotic-left)
strf = roi_strf(FULL_MASKED, 'gammatone_on_overt_8', 'dichotic-left')
p = plot.Array(strf * 1e3, title='Overt onset attend STRF (dichotic-left)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset ignore STRF (dichotic-left)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'dichotic-left')
p = plot.Array(strf * 1e3, title='Overt onset ignore STRF (dichotic-left)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset attend peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_MASKED, 'gammatone_on_overt_8', 'dichotic-left')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset attend (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset attend (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak amplitude (dichotic-left)')

In [ ]:
# Overt onset ignore peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'dichotic-left')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset ignore (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset ignore (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak amplitude (dichotic-left)')

In [ ]:
# Overt onset attention effect: attend - ignore (dichotic-left)
data = e.load_trfs(-1, FULL_MASKED, epoch='dichotic-left', **PARAMETERS)
attend_trf = data['gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Overt onset attention effect: attend - ignore (dichotic-left)')

## Dichotic Left (masked onset)

In [ ]:
# Masked onset attend unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: attended masked onsets unique contribution, t = 8.42, p < .001 - the strongest of the four masking predictors.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(ATTEND), 'dichotic-left')
show_topomap_trio(data, result, 'Masked onset attend', 'dichotic-left', masked_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset attend (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset attend', 'dichotic-left')

In [ ]:
# Masked onset ignore unique contribution (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: ignored masked onsets unique contribution, t = 5.23, p < .001 - the brain recovers masked information even for the ignored speaker.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(IGNORE), 'dichotic-left')
show_topomap_trio(data, result, 'Masked onset ignore', 'dichotic-left', masked_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset ignore (dichotic-left): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset ignore', 'dichotic-left')

In [ ]:
# Masked onset attend STRF (dichotic-left)
strf = roi_strf(FULL_MASKED, 'gammatone_on_masked_8', 'dichotic-left')
p = plot.Array(strf * 1e3, title='Masked onset attend STRF (dichotic-left)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset ignore STRF (dichotic-left)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'dichotic-left')
p = plot.Array(strf * 1e3, title='Masked onset ignore STRF (dichotic-left)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset attend peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_MASKED, 'gammatone_on_masked_8', 'dichotic-left')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset attend (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset attend (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak amplitude (dichotic-left)')

In [ ]:
# Masked onset ignore peak amplitude & timing (dichotic-left)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'dichotic-left')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset ignore (dichotic-left) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset ignore (dichotic-left) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak time (dichotic-left)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak amplitude (dichotic-left)')

In [ ]:
# Masked onset attention effect: attend - ignore (dichotic-left)
data = e.load_trfs(-1, FULL_MASKED, epoch='dichotic-left', **PARAMETERS)
attend_trf = data['gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Masked onset attention effect: attend - ignore (dichotic-left)')

In [ ]:
# Overt vs. masked peak comparison, attend (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: attended overt 72ms vs. attended masked 91ms (t=2.85,p=.009); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'gammatone_on_overt_8'), ('masked', 'gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'dichotic-left')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='attend positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='attend negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='attend positive peak amplitude: overt vs. masked'))

In [ ]:
# Overt vs. masked peak comparison, ignore (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: ignored overt 83ms vs. ignored masked 97ms (t=6.11,p<.001); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'bg_gammatone_on_overt_8'), ('masked', 'bg_gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'dichotic-left')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='ignore positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='ignore negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='ignore positive peak amplitude: overt vs. masked'))

In [ ]:
# Stream x masking interaction on peak amplitude (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: stream x masking interaction, F(1,25) = 24.45, p < .001 - overt onsets show no early attention effect, masked onsets do.
dss = []
for stream_name, is_attend in (('attend', True), ('ignore', False)):
    for masking_name in ('overt', 'masked'):
        prefix = '' if is_attend else 'bg_'
        column = f'{prefix}gammatone_on_{masking_name}_8'
        ds = roi_trf(FULL_MASKED, column, 'dichotic-left')
        ds[:, 'stream'] = stream_name
        ds[:, 'masking'] = masking_name
        dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
display(test.ANOVA('positive_amp', 'stream * masking * subject', data=data, title='stream x masking interaction (dichotic-left)'))

## Dichotic Left (intensity-level control)

In [ ]:
# Masked-aware vs. level-aware (dichotic-left)
# Brodbeck et al. 2020 ran this comparison too: the ignored-speaker-aware model still beat the loudness-only model, t = 9.21, p < .001 - the masking effect isn't just about volume.
data, diff, result = compute_direct_comparison(FULL_MASKED, LEVEL_AWARE, 'dichotic-left')
show_topomap_trio(data, result, 'Masked-aware vs. level-aware', 'dichotic-left', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked-aware vs. level-aware (dichotic-left): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked-aware vs. level-aware', 'dichotic-left')

## Dichotic Right

In [ ]:
# Onset attend unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: attended onsets unique contribution, t = 6.32, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(ATTEND), 'dichotic-right')
show_topomap_trio(data, result, 'Onset attend', 'dichotic-right', onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset attend (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset attend', 'dichotic-right')

In [ ]:
# Onset ignore unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: ignored onsets unique contribution, t = 6.70, p < .001 - the ignored speaker is tracked separately.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(IGNORE), 'dichotic-right')
show_topomap_trio(data, result, 'Onset ignore', 'dichotic-right', onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset ignore (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset ignore', 'dichotic-right')

In [ ]:
# Onset mix unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: mixture onsets unique contribution, t = 8.61, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(MIX), 'dichotic-right')
show_topomap_trio(data, result, 'Onset mix', 'dichotic-right', onset(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset mix (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset mix', 'dichotic-right')

In [ ]:
# Envelope attend unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: attended envelope unique contribution, t = 7.37, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(ATTEND), 'dichotic-right')
show_topomap_trio(data, result, 'Envelope attend', 'dichotic-right', envelope(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope attend (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope attend', 'dichotic-right')

In [ ]:
# Envelope ignore unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: ignored envelope unique contribution, t = 6.28, p < .001 - the ignored speaker's envelope is tracked separately too.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(IGNORE), 'dichotic-right')
show_topomap_trio(data, result, 'Envelope ignore', 'dichotic-right', envelope(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope ignore (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope ignore', 'dichotic-right')

In [ ]:
# Envelope mix unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: mixture envelope unique contribution, t = 5.70, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(MIX), 'dichotic-right')
show_topomap_trio(data, result, 'Envelope mix', 'dichotic-right', envelope(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope mix (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope mix', 'dichotic-right')

In [ ]:
# Mix source STRF: envelope + onset (dichotic-right)
envelope_strf = roi_strf(FULL_BASE, 'mix_gammatone_8', 'dichotic-right')
onset_strf = roi_strf(FULL_BASE, 'mix_gammatone_on_8', 'dichotic-right')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Mix source STRF (dichotic-right)', xlim=(-0.050, 0.550))

In [ ]:
# Attend source STRF: envelope + onset (dichotic-right)
envelope_strf = roi_strf(FULL_BASE, 'gammatone_8', 'dichotic-right')
onset_strf = roi_strf(FULL_BASE, 'gammatone_on_8', 'dichotic-right')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Attend source STRF (dichotic-right)', xlim=(-0.050, 0.550))

In [ ]:
# Ignore source STRF: envelope + onset (dichotic-right)
envelope_strf = roi_strf(FULL_BASE, 'bg_gammatone_8', 'dichotic-right')
onset_strf = roi_strf(FULL_BASE, 'bg_gammatone_on_8', 'dichotic-right')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Ignore source STRF (dichotic-right)', xlim=(-0.050, 0.550))

In [ ]:
# Onset attend peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_BASE, 'gammatone_on_8', 'dichotic-right')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset attend (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset attend (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak amplitude (dichotic-right)')

In [ ]:
# Onset ignore peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_BASE, 'bg_gammatone_on_8', 'dichotic-right')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset ignore (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset ignore (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak amplitude (dichotic-right)')

In [ ]:
# Onset mix peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_BASE, 'mix_gammatone_on_8', 'dichotic-right')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset mix (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset mix (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak amplitude (dichotic-right)')

In [ ]:
# Envelope attend peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_BASE, 'gammatone_8', 'dichotic-right')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope attend (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope attend (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak amplitude (dichotic-right)')

In [ ]:
# Envelope ignore peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_BASE, 'bg_gammatone_8', 'dichotic-right')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope ignore (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope ignore (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak amplitude (dichotic-right)')

In [ ]:
# Envelope mix peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_BASE, 'mix_gammatone_8', 'dichotic-right')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope mix (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope mix (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak amplitude (dichotic-right)')

In [ ]:
# Onset peak comparison across mix, attend, and ignore (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: positive peak latency mixture 72ms, attended 81ms (t=4.47,p<.001), ignored 89ms (t=6.92,p<.001); negative peak latency mixture 138ms, attended 150ms (t=3.20,p=.004); positive peak amplitude bigger for mixture than attended (t=8.41) or ignored (t=7.66).
dss = []
for role_name, column in (('mix', 'mix_gammatone_on_8'), ('attend', 'gammatone_on_8'), ('ignore', 'bg_gammatone_on_8')):
    ds = roi_trf(FULL_BASE, column, 'dichotic-right')
    ds[:, 'role'] = role_name
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
# The onset TRF waveform mix/attend/ignore were each just measured
# from, overlaid so their shape and relative timing are visible
# directly, not just as extracted peak numbers below.
p = plot.UTSStat('trf_sm*1e3', 'role', data=data, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550),
                  title='Onset TRF: mix vs. attend vs. ignore (dichotic-right)', labels=LABELS_ROLE, colors=COLORS_ROLE)

In [ ]:
display(test.pairwise('positive_time', 'role', match='subject', data=data, corr=False, title='positive peak latency: mix vs. attend vs. ignore'))
display(test.pairwise('negative_time', 'role', match='subject', data=data, corr=False, title='negative peak latency: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak latency (dichotic-right)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='negative peak latency (dichotic-right)')
p.set_xtick_rotation(30)

In [ ]:
display(test.pairwise('positive_amp', 'role', match='subject', data=data, corr=False, title='positive peak amplitude: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak amplitude (dichotic-right)')
p.set_xtick_rotation(30)

In [ ]:
# Envelope attention effect: attend - ignore (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: envelope attention-effect difference wave: negative bump ~100ms, positive bump ~200ms, starting almost immediately.
data = e.load_trfs(-1, FULL_BASE, epoch='dichotic-right', **PARAMETERS)
attend_trf = data['gammatone_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Envelope attention effect: attend - ignore (dichotic-right)')

In [ ]:
# Onset attention effect: attend - ignore (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: onset attention effect appears later than the envelope's, once the masked-onset response develops (see the masked-onset section below).
data = e.load_trfs(-1, FULL_BASE, epoch='dichotic-right', **PARAMETERS)
attend_trf = data['gammatone_on_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Onset attention effect: attend - ignore (dichotic-right)')

## Dichotic Right (overt onsets)

In [ ]:
# Overt/masked split vs. plain onsets (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: splitting onsets into overt/masked significantly improved the model overall, t = 6.81, p < .001.
data, diff, result = compute_direct_comparison(FULL_MASKED, FULL_BASE, 'dichotic-right')
show_topomap_trio(data, result, 'Overt/masked split vs. plain onsets', 'dichotic-right', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt/masked split vs. plain onsets (dichotic-right): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt/masked split vs. plain onsets', 'dichotic-right')

In [ ]:
# Overt onset attend unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: attended overt onsets unique contribution, t = 3.34, p = .027.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(ATTEND), 'dichotic-right')
show_topomap_trio(data, result, 'Overt onset attend', 'dichotic-right', overt_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset attend (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset attend', 'dichotic-right')

In [ ]:
# Overt onset ignore unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: ignored overt onsets unique contribution, t = 3.82, p = .016.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(IGNORE), 'dichotic-right')
show_topomap_trio(data, result, 'Overt onset ignore', 'dichotic-right', overt_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset ignore (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset ignore', 'dichotic-right')

In [ ]:
# Overt onset attend STRF (dichotic-right)
strf = roi_strf(FULL_MASKED, 'gammatone_on_overt_8', 'dichotic-right')
p = plot.Array(strf * 1e3, title='Overt onset attend STRF (dichotic-right)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset ignore STRF (dichotic-right)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'dichotic-right')
p = plot.Array(strf * 1e3, title='Overt onset ignore STRF (dichotic-right)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset attend peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_MASKED, 'gammatone_on_overt_8', 'dichotic-right')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset attend (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset attend (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak amplitude (dichotic-right)')

In [ ]:
# Overt onset ignore peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'dichotic-right')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset ignore (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset ignore (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak amplitude (dichotic-right)')

In [ ]:
# Overt onset attention effect: attend - ignore (dichotic-right)
data = e.load_trfs(-1, FULL_MASKED, epoch='dichotic-right', **PARAMETERS)
attend_trf = data['gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Overt onset attention effect: attend - ignore (dichotic-right)')

## Dichotic Right (masked onset)

In [ ]:
# Masked onset attend unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: attended masked onsets unique contribution, t = 8.42, p < .001 - the strongest of the four masking predictors.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(ATTEND), 'dichotic-right')
show_topomap_trio(data, result, 'Masked onset attend', 'dichotic-right', masked_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset attend (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset attend', 'dichotic-right')

In [ ]:
# Masked onset ignore unique contribution (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: ignored masked onsets unique contribution, t = 5.23, p < .001 - the brain recovers masked information even for the ignored speaker.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(IGNORE), 'dichotic-right')
show_topomap_trio(data, result, 'Masked onset ignore', 'dichotic-right', masked_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset ignore (dichotic-right): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset ignore', 'dichotic-right')

In [ ]:
# Masked onset attend STRF (dichotic-right)
strf = roi_strf(FULL_MASKED, 'gammatone_on_masked_8', 'dichotic-right')
p = plot.Array(strf * 1e3, title='Masked onset attend STRF (dichotic-right)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset ignore STRF (dichotic-right)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'dichotic-right')
p = plot.Array(strf * 1e3, title='Masked onset ignore STRF (dichotic-right)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset attend peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_MASKED, 'gammatone_on_masked_8', 'dichotic-right')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset attend (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset attend (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak amplitude (dichotic-right)')

In [ ]:
# Masked onset ignore peak amplitude & timing (dichotic-right)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'dichotic-right')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset ignore (dichotic-right) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset ignore (dichotic-right) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak time (dichotic-right)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak amplitude (dichotic-right)')

In [ ]:
# Masked onset attention effect: attend - ignore (dichotic-right)
data = e.load_trfs(-1, FULL_MASKED, epoch='dichotic-right', **PARAMETERS)
attend_trf = data['gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Masked onset attention effect: attend - ignore (dichotic-right)')

In [ ]:
# Overt vs. masked peak comparison, attend (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: attended overt 72ms vs. attended masked 91ms (t=2.85,p=.009); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'gammatone_on_overt_8'), ('masked', 'gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'dichotic-right')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='attend positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='attend negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='attend positive peak amplitude: overt vs. masked'))

In [ ]:
# Overt vs. masked peak comparison, ignore (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: ignored overt 83ms vs. ignored masked 97ms (t=6.11,p<.001); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'bg_gammatone_on_overt_8'), ('masked', 'bg_gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'dichotic-right')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='ignore positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='ignore negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='ignore positive peak amplitude: overt vs. masked'))

In [ ]:
# Stream x masking interaction on peak amplitude (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: stream x masking interaction, F(1,25) = 24.45, p < .001 - overt onsets show no early attention effect, masked onsets do.
dss = []
for stream_name, is_attend in (('attend', True), ('ignore', False)):
    for masking_name in ('overt', 'masked'):
        prefix = '' if is_attend else 'bg_'
        column = f'{prefix}gammatone_on_{masking_name}_8'
        ds = roi_trf(FULL_MASKED, column, 'dichotic-right')
        ds[:, 'stream'] = stream_name
        ds[:, 'masking'] = masking_name
        dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
display(test.ANOVA('positive_amp', 'stream * masking * subject', data=data, title='stream x masking interaction (dichotic-right)'))

## Dichotic Right (intensity-level control)

In [ ]:
# Masked-aware vs. level-aware (dichotic-right)
# Brodbeck et al. 2020 ran this comparison too: the ignored-speaker-aware model still beat the loudness-only model, t = 9.21, p < .001 - the masking effect isn't just about volume.
data, diff, result = compute_direct_comparison(FULL_MASKED, LEVEL_AWARE, 'dichotic-right')
show_topomap_trio(data, result, 'Masked-aware vs. level-aware', 'dichotic-right', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked-aware vs. level-aware (dichotic-right): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked-aware vs. level-aware', 'dichotic-right')

## Binaural

In [ ]:
# Onset attend unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended onsets unique contribution, t = 6.32, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(ATTEND), 'binaural')
show_topomap_trio(data, result, 'Onset attend', 'binaural', onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset attend (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset attend', 'binaural')

In [ ]:
# Onset ignore unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored onsets unique contribution, t = 6.70, p < .001 - the ignored speaker is tracked separately.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(IGNORE), 'binaural')
show_topomap_trio(data, result, 'Onset ignore', 'binaural', onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset ignore (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset ignore', 'binaural')

In [ ]:
# Onset mix unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: mixture onsets unique contribution, t = 8.61, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, onset(MIX), 'binaural')
show_topomap_trio(data, result, 'Onset mix', 'binaural', onset(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset mix (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Onset mix', 'binaural')

In [ ]:
# Envelope attend unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended envelope unique contribution, t = 7.37, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(ATTEND), 'binaural')
show_topomap_trio(data, result, 'Envelope attend', 'binaural', envelope(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope attend (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope attend', 'binaural')

In [ ]:
# Envelope ignore unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored envelope unique contribution, t = 6.28, p < .001 - the ignored speaker's envelope is tracked separately too.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(IGNORE), 'binaural')
show_topomap_trio(data, result, 'Envelope ignore', 'binaural', envelope(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope ignore (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope ignore', 'binaural')

In [ ]:
# Envelope mix unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: mixture envelope unique contribution, t = 5.70, p < .001.
data, diff, result = compute_unique_contribution(FULL_BASE, envelope(MIX), 'binaural')
show_topomap_trio(data, result, 'Envelope mix', 'binaural', envelope(MIX))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope mix (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Envelope mix', 'binaural')

In [ ]:
# Mix source STRF: envelope + onset (binaural)
envelope_strf = roi_strf(FULL_BASE, 'mix_gammatone_8', 'binaural')
onset_strf = roi_strf(FULL_BASE, 'mix_gammatone_on_8', 'binaural')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Mix source STRF (binaural)', xlim=(-0.050, 0.550))

In [ ]:
# Attend source STRF: envelope + onset (binaural)
envelope_strf = roi_strf(FULL_BASE, 'gammatone_8', 'binaural')
onset_strf = roi_strf(FULL_BASE, 'gammatone_on_8', 'binaural')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Attend source STRF (binaural)', xlim=(-0.050, 0.550))

In [ ]:
# Ignore source STRF: envelope + onset (binaural)
envelope_strf = roi_strf(FULL_BASE, 'bg_gammatone_8', 'binaural')
onset_strf = roi_strf(FULL_BASE, 'bg_gammatone_on_8', 'binaural')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
                title='Ignore source STRF (binaural)', xlim=(-0.050, 0.550))

In [ ]:
# Onset attend peak amplitude & timing (binaural)
data = roi_trf(FULL_BASE, 'gammatone_on_8', 'binaural')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset attend (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset attend (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset attend positive peak amplitude (binaural)')

In [ ]:
# Onset ignore peak amplitude & timing (binaural)
data = roi_trf(FULL_BASE, 'bg_gammatone_on_8', 'binaural')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset ignore (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset ignore (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset ignore positive peak amplitude (binaural)')

In [ ]:
# Onset mix peak amplitude & timing (binaural)
data = roi_trf(FULL_BASE, 'mix_gammatone_on_8', 'binaural')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset mix (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset mix (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Onset mix positive peak amplitude (binaural)')

In [ ]:
# Envelope attend peak amplitude & timing (binaural)
data = roi_trf(FULL_BASE, 'gammatone_8', 'binaural')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope attend (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope attend (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope attend positive peak amplitude (binaural)')

In [ ]:
# Envelope ignore peak amplitude & timing (binaural)
data = roi_trf(FULL_BASE, 'bg_gammatone_8', 'binaural')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope ignore (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope ignore (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope ignore positive peak amplitude (binaural)')

In [ ]:
# Envelope mix peak amplitude & timing (binaural)
data = roi_trf(FULL_BASE, 'mix_gammatone_8', 'binaural')
data = add_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope mix (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope mix (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Envelope mix positive peak amplitude (binaural)')

In [ ]:
# Onset peak comparison across mix, attend, and ignore (binaural)
# Brodbeck et al. 2020 ran this comparison too: positive peak latency mixture 72ms, attended 81ms (t=4.47,p<.001), ignored 89ms (t=6.92,p<.001); negative peak latency mixture 138ms, attended 150ms (t=3.20,p=.004); positive peak amplitude bigger for mixture than attended (t=8.41) or ignored (t=7.66).
dss = []
for role_name, column in (('mix', 'mix_gammatone_on_8'), ('attend', 'gammatone_on_8'), ('ignore', 'bg_gammatone_on_8')):
    ds = roi_trf(FULL_BASE, column, 'binaural')
    ds[:, 'role'] = role_name
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
# The onset TRF waveform mix/attend/ignore were each just measured
# from, overlaid so their shape and relative timing are visible
# directly, not just as extracted peak numbers below.
p = plot.UTSStat('trf_sm*1e3', 'role', data=data, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550),
                  title='Onset TRF: mix vs. attend vs. ignore (binaural)', labels=LABELS_ROLE, colors=COLORS_ROLE)

In [ ]:
display(test.pairwise('positive_time', 'role', match='subject', data=data, corr=False, title='positive peak latency: mix vs. attend vs. ignore'))
display(test.pairwise('negative_time', 'role', match='subject', data=data, corr=False, title='negative peak latency: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak latency (binaural)')
p.set_xtick_rotation(30)
p = plot.Barplot('negative_time', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='negative peak latency (binaural)')
p.set_xtick_rotation(30)

In [ ]:
display(test.pairwise('positive_amp', 'role', match='subject', data=data, corr=False, title='positive peak amplitude: mix vs. attend vs. ignore'))

In [ ]:
p = plot.Barplot('positive_amp', 'role', match='subject', data=data, h=2.5, w=2.5, labels=LABELS_ROLE, corr=False, title='positive peak amplitude (binaural)')
p.set_xtick_rotation(30)

In [ ]:
# Envelope attention effect: attend - ignore (binaural)
# Brodbeck et al. 2020 ran this comparison too: envelope attention-effect difference wave: negative bump ~100ms, positive bump ~200ms, starting almost immediately.
data = e.load_trfs(-1, FULL_BASE, epoch='binaural', **PARAMETERS)
attend_trf = data['gammatone_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Envelope attention effect: attend - ignore (binaural)')

In [ ]:
# Onset attention effect: attend - ignore (binaural)
# Brodbeck et al. 2020 ran this comparison too: onset attention effect appears later than the envelope's, once the masked-onset response develops (see the masked-onset section below).
data = e.load_trfs(-1, FULL_BASE, epoch='binaural', **PARAMETERS)
attend_trf = data['gammatone_on_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Onset attention effect: attend - ignore (binaural)')

## Binaural (overt onsets)

In [ ]:
# Overt/masked split vs. plain onsets (binaural)
# Brodbeck et al. 2020 ran this comparison too: splitting onsets into overt/masked significantly improved the model overall, t = 6.81, p < .001.
data, diff, result = compute_direct_comparison(FULL_MASKED, FULL_BASE, 'binaural')
show_topomap_trio(data, result, 'Overt/masked split vs. plain onsets', 'binaural', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt/masked split vs. plain onsets (binaural): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt/masked split vs. plain onsets', 'binaural')

In [ ]:
# Overt onset attend unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended overt onsets unique contribution, t = 3.34, p = .027.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(ATTEND), 'binaural')
show_topomap_trio(data, result, 'Overt onset attend', 'binaural', overt_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset attend (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset attend', 'binaural')

In [ ]:
# Overt onset ignore unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored overt onsets unique contribution, t = 3.82, p = .016.
data, diff, result = compute_unique_contribution(FULL_MASKED, overt_onset(IGNORE), 'binaural')
show_topomap_trio(data, result, 'Overt onset ignore', 'binaural', overt_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Overt onset ignore (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Overt onset ignore', 'binaural')

In [ ]:
# Overt onset attend STRF (binaural)
strf = roi_strf(FULL_MASKED, 'gammatone_on_overt_8', 'binaural')
p = plot.Array(strf * 1e3, title='Overt onset attend STRF (binaural)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset ignore STRF (binaural)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'binaural')
p = plot.Array(strf * 1e3, title='Overt onset ignore STRF (binaural)', xlim=(-0.050, 0.550))

In [ ]:
# Overt onset attend peak amplitude & timing (binaural)
data = roi_trf(FULL_MASKED, 'gammatone_on_overt_8', 'binaural')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset attend (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset attend (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset attend positive peak amplitude (binaural)')

In [ ]:
# Overt onset ignore peak amplitude & timing (binaural)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_overt_8', 'binaural')
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Overt onset ignore (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Overt onset ignore (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Overt onset ignore positive peak amplitude (binaural)')

In [ ]:
# Overt onset attention effect: attend - ignore (binaural)
data = e.load_trfs(-1, FULL_MASKED, epoch='binaural', **PARAMETERS)
attend_trf = data['gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_overt_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Overt onset attention effect: attend - ignore (binaural)')

## Binaural (masked onset)

In [ ]:
# Masked onset attend unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended masked onsets unique contribution, t = 8.42, p < .001 - the strongest of the four masking predictors.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(ATTEND), 'binaural')
show_topomap_trio(data, result, 'Masked onset attend', 'binaural', masked_onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset attend (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset attend', 'binaural')

In [ ]:
# Masked onset ignore unique contribution (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored masked onsets unique contribution, t = 5.23, p < .001 - the brain recovers masked information even for the ignored speaker.
data, diff, result = compute_unique_contribution(FULL_MASKED, masked_onset(IGNORE), 'binaural')
show_topomap_trio(data, result, 'Masked onset ignore', 'binaural', masked_onset(IGNORE))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked onset ignore (binaural): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked onset ignore', 'binaural')

In [ ]:
# Masked onset attend STRF (binaural)
strf = roi_strf(FULL_MASKED, 'gammatone_on_masked_8', 'binaural')
p = plot.Array(strf * 1e3, title='Masked onset attend STRF (binaural)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset ignore STRF (binaural)
strf = roi_strf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'binaural')
p = plot.Array(strf * 1e3, title='Masked onset ignore STRF (binaural)', xlim=(-0.050, 0.550))

In [ ]:
# Masked onset attend peak amplitude & timing (binaural)
data = roi_trf(FULL_MASKED, 'gammatone_on_masked_8', 'binaural')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset attend (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset attend (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset attend positive peak amplitude (binaural)')

In [ ]:
# Masked onset ignore peak amplitude & timing (binaural)
data = roi_trf(FULL_MASKED, 'bg_gammatone_on_masked_8', 'binaural')
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Masked onset ignore (binaural) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Masked onset ignore (binaural) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
p = plot.Barplot('positive_time', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak time (binaural)')
p = plot.Barplot('positive_amp', match='subject', data=data, h=2.5, w=2, corr=False, title='Masked onset ignore positive peak amplitude (binaural)')

In [ ]:
# Masked onset attention effect: attend - ignore (binaural)
data = e.load_trfs(-1, FULL_MASKED, epoch='binaural', **PARAMETERS)
attend_trf = data['gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
ignore_trf = data['bg_gammatone_on_masked_8'].sum('frequency').mean(sensor=ROI)
attention_effect = attend_trf - ignore_trf
p = plot.UTSStat(attention_effect * 1e3, frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), title='Masked onset attention effect: attend - ignore (binaural)')

In [ ]:
# Overt vs. masked peak comparison, attend (binaural)
# Brodbeck et al. 2020 ran this comparison too: attended overt 72ms vs. attended masked 91ms (t=2.85,p=.009); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'gammatone_on_overt_8'), ('masked', 'gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'binaural')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='attend positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='attend negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='attend positive peak amplitude: overt vs. masked'))

In [ ]:
# Overt vs. masked peak comparison, ignore (binaural)
# Brodbeck et al. 2020 ran this comparison too: ignored overt 83ms vs. ignored masked 97ms (t=6.11,p<.001); combined (not stream-split) late negative peak, overt 133ms vs. masked 182ms (t=4.45,p<.001).
dss = []
for masking, column in (('overt', 'bg_gammatone_on_overt_8'), ('masked', 'bg_gammatone_on_masked_8')):
    ds = roi_trf(FULL_MASKED, column, 'binaural')
    ds[:, 'masking'] = masking
    dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', MASKED_NEGATIVE_PEAK_WINDOW, kind='min')

In [ ]:
display(test.pairwise('positive_time', 'masking', match='subject', data=data, corr=False, title='ignore positive peak latency: overt vs. masked'))
display(test.pairwise('negative_time', 'masking', match='subject', data=data, corr=False, title='ignore negative peak latency: overt vs. masked'))

In [ ]:
display(test.pairwise('positive_amp', 'masking', match='subject', data=data, corr=False, title='ignore positive peak amplitude: overt vs. masked'))

In [ ]:
# Stream x masking interaction on peak amplitude (binaural)
# Brodbeck et al. 2020 ran this comparison too: stream x masking interaction, F(1,25) = 24.45, p < .001 - overt onsets show no early attention effect, masked onsets do.
dss = []
for stream_name, is_attend in (('attend', True), ('ignore', False)):
    for masking_name in ('overt', 'masked'):
        prefix = '' if is_attend else 'bg_'
        column = f'{prefix}gammatone_on_{masking_name}_8'
        ds = roi_trf(FULL_MASKED, column, 'binaural')
        ds[:, 'stream'] = stream_name
        ds[:, 'masking'] = masking_name
        dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', MASKED_POSITIVE_PEAK_WINDOW, kind='max')
display(test.ANOVA('positive_amp', 'stream * masking * subject', data=data, title='stream x masking interaction (binaural)'))

## Binaural (intensity-level control)

In [ ]:
# Masked-aware vs. level-aware (binaural)
# Brodbeck et al. 2020 ran this comparison too: the ignored-speaker-aware model still beat the loudness-only model, t = 9.21, p < .001 - the masking effect isn't just about volume.
data, diff, result = compute_direct_comparison(FULL_MASKED, LEVEL_AWARE, 'binaural')
show_topomap_trio(data, result, 'Masked-aware vs. level-aware', 'binaural', None)

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Masked-aware vs. level-aware (binaural): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_bar(diff, 'Masked-aware vs. level-aware', 'binaural')

## Cross-condition comparisons

Every analysis above stays within one listening condition. This section
asks the complementary question: for a *given* predictor (mixture,
attend, or ignore), does its onset peak's timing or amplitude change
depending on which of the four spatial conditions it's in? That means
pooling diotic, dichotic-left, dichotic-right, and binaural together -
the one place in this notebook that isn't self-contained within a single
condition's section, since the comparison is *across* conditions by
definition.

In [ ]:
# Onset peak time/amplitude, pooled across all four two-talker conditions -
# same roi_trf()/add_peak() building blocks used throughout, just looped
# over epoch as well as role this time.
dss = []
for epoch in ('diotic', 'dichotic-left', 'dichotic-right', 'binaural'):
    for role_name, column in (('mix', 'mix_gammatone_on_8'), ('attend', 'gammatone_on_8'), ('ignore', 'bg_gammatone_on_8')):
        ds = roi_trf(FULL_BASE, column, epoch)
        ds[:, 'role'] = role_name
        ds[:, 'epoch'] = epoch
        dss.append(ds)
data = combine(dss)
data = add_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')

### Peak time by predictor

In [ ]:
# Attend onset positive peak latency across conditions
display(test.pairwise('positive_time', 'epoch', match='subject', sub="role == 'attend'", data=data, corr=False, title='Attend positive peak latency across conditions'))

In [ ]:
p = plot.Barplot('positive_time', 'epoch', match='subject', sub="role == 'attend'", data=data, h=2, w=2.5, title='Attend', corr=False)
p.set_xtick_rotation(30)

In [ ]:
# Ignore onset positive peak latency across conditions
display(test.pairwise('positive_time', 'epoch', match='subject', sub="role == 'ignore'", data=data, corr=False, title='Ignore positive peak latency across conditions'))

In [ ]:
p = plot.Barplot('positive_time', 'epoch', match='subject', sub="role == 'ignore'", data=data, h=2, w=2.5, title='Ignore', corr=False)
p.set_xtick_rotation(30)

In [ ]:
# Mixture onset positive peak latency across conditions
display(test.pairwise('positive_time', 'epoch', match='subject', sub="role == 'mix'", data=data, corr=False, title='Mixture positive peak latency across conditions'))

In [ ]:
p = plot.Barplot('positive_time', 'epoch', match='subject', sub="role == 'mix'", data=data, h=2, w=2.5, title='Mixture', corr=False)
p.set_xtick_rotation(30)

### Peak amplitude by predictor

In [ ]:
# Attend onset positive peak amplitude across conditions
display(test.pairwise('positive_amp', 'epoch', match='subject', sub="role == 'attend'", data=data, corr=False, title='Attend positive peak amplitude across conditions'))

In [ ]:
p = plot.Barplot('positive_amp', 'epoch', match='subject', sub="role == 'attend'", data=data, h=2, w=2.5, title='Attend', corr=False)
p.set_xtick_rotation(30)

In [ ]:
# Ignore onset positive peak amplitude across conditions
display(test.pairwise('positive_amp', 'epoch', match='subject', sub="role == 'ignore'", data=data, corr=False, title='Ignore positive peak amplitude across conditions'))

In [ ]:
p = plot.Barplot('positive_amp', 'epoch', match='subject', sub="role == 'ignore'", data=data, h=2, w=2.5, title='Ignore', corr=False)
p.set_xtick_rotation(30)

In [ ]:
# Mixture onset positive peak amplitude across conditions
display(test.pairwise('positive_amp', 'epoch', match='subject', sub="role == 'mix'", data=data, corr=False, title='Mixture positive peak amplitude across conditions'))

In [ ]:
p = plot.Barplot('positive_amp', 'epoch', match='subject', sub="role == 'mix'", data=data, h=2, w=2.5, title='Mixture', corr=False)
p.set_xtick_rotation(30)